<a href="https://colab.research.google.com/github/sanmeshh/pytorch_learning/blob/9.ANN_Optune/ann_optuna_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset,DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [2]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device:{device}')

device:cpu


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
#reproducibility
torch.manual_seed(108)

In [6]:
df=pd.read_csv('/content/drive/MyDrive/fashion-mnist_train.csv')

In [7]:
X_train=df.iloc[:,1:].values
y_train=df.iloc[:,0].values

In [8]:
df2=pd.read_csv('/content/drive/MyDrive/fashion-mnist_test.csv')
X_test=df2.iloc[:,1:].values
y_test=df2.iloc[:,0].values


In [9]:
#scaling the features values are like 0,142,213
X_train=X_train/255.0
X_test=X_test/255.0

In [10]:
X_train.shape

(60000, 784)

In [27]:
#custom dataset
class CustomDataset(Dataset):
  def __init__(self,features,labels):
    self.features=torch.tensor(features,dtype=torch.float32)#remember float for features
    self.labels=torch.tensor(labels,dtype=torch.long)#and long for labels
  def __len__(self):
    return len(self.features)
  def __getitem__(self,index):
    return self.features[index],self.labels[index]


In [28]:
#create train_dataset object
train_dataset=CustomDataset(X_train,y_train)


In [29]:
len(train_dataset)

60000

In [ ]:
train_dataset[3]

(tensor([0.0000, 0.0000, 0.0000, 0.0039, 0.0078, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.4471, 0.7176, 0.4392, 0.2157, 0.0902, 0.2824, 0.4000, 0.6471,
         0.6275, 0.1098, 0.0000, 0.0000, 0.0000, 0.0039, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0039, 0.0000, 0.0000, 0.0941,
         0.7373, 0.6392, 0.3647, 0.5333, 0.6000, 0.6588, 0.9882, 0.6824, 0.5333,
         0.6510, 0.5098, 0.4824, 0.5137, 0.2588, 0.0000, 0.0000, 0.0039, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0078, 0.0000, 0.0392, 0.6157,
         0.8471, 0.8863, 0.8157, 0.5569, 0.2588, 0.4510, 0.5843, 0.9020, 0.7451,
         0.7686, 0.7765, 0.6745, 0.8706, 0.4196, 0.6471, 0.8275, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.4627,
         0.8392, 0.6824, 0.6588, 0.4275, 0.7843, 0.4863, 0.5882, 0.5608, 0.2275,
         0.2471, 0.3490, 0.5373, 0.3804, 0.6588, 0.5412, 0.5569, 0.7647, 0.6118,
         0.0000, 0.0000, 0.0

In [30]:
test_dataset=CustomDataset(X_test,y_test)

In [31]:
#dividing data into batches
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=False,pin_memory=True)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False,pin_memory=True)

In [45]:
class MyNN(nn.Module):

  def __init__(self, input_dim , output_dim , num_hidden_layers , neurons_per_layer , dropout_rate):

    super().__init__()

    layers=[]

    for i in range(num_hidden_layers):

      layers.append(nn.Linear(input_dim,neurons_per_layer))
      layers.append(nn.BatchNorm1d(neurons_per_layer))
      layers.append(nn.ReLU())
      layers.append(nn.Dropout(dropout_rate))
      input_dim=neurons_per_layer

    layers.append(nn.Linear(neurons_per_layer,output_dim))
  #* unpack list elements
    self.model=nn.Sequential(*layers)

  def forward(self,X):
    return self.model(X)









In [48]:
#objective function
def objective(trial):

  #next hyperparameter values from the search space
  num_hidden_layers=trial.suggest_int("Hidden_layers",1,5)
  neurons_per_layer=trial.suggest_int("neurons_per_layer",8,128,step=8)
  epochs = trial.suggest_int("epochs", 10, 50, step=10)
  lr= trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
  dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5, step=0.1)
  batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128])
  optimizer_name = trial.suggest_categorical("optimizer", ['Adam', 'SGD', 'RMSprop'])
  weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)


  #model init
  input_dim=784
  output_dim=10

  model=MyNN(input_dim,output_dim,num_hidden_layers,neurons_per_layer,dropout_rate)
  model.to(device)


  #optimizer selection
  criterion=nn.CrossEntropyLoss()
  optimizer=optim.SGD(model.parameters(),lr=lr,weight_decay=1e-4)
  if optimizer_name == 'Adam':
      optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
  elif optimizer_name == 'SGD':
      optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)
  else:
       optim.RMSprop(model.parameters(), lr=lr, weight_decay=weight_decay)




  #training loop

  for epoch in range(epochs):

    for batch_features,batch_labels in train_loader:

      #move data to gpu
      batch_features,batch_labels=batch_features.to(device),batch_labels.to(device)

      #forward pass
      outputs=model(batch_features)

      #loss
      loss=criterion(outputs,batch_labels)

      #backward pass
      optimizer.zero_grad()
      loss.backward()


      #update grads
      optimizer.step()

  model.eval()
  total=0
  correct=0

  #we dont want to calculate gradients during evaluation
  with torch.no_grad():

    for batch_features,batch_labels in test_loader:
      batch_features,batch_labels=batch_features.to(device),batch_labels.to(device)

      outputs=model(batch_features)


      _ , predicted=torch.max(outputs,1)

      #getting the total no of samples by adding samples of each batch
      total=total+batch_labels.shape[0]
      # print(total)

      correct+=(predicted==batch_labels).sum().item()

    accuracy=(correct/total)



  return accuracy



In [24]:
!pip install optuna

In [49]:
import optuna
study=optuna.create_study(direction='maximize')

[I 2025-04-14 05:35:22,948] A new study created in memory with name: no-name-ad4dae5d-e23e-4c31-a071-25b413284f26


In [50]:
study.optimize(objective,n_trials=10)

[I 2025-04-14 05:36:04,850] Trial 0 finished with value: 0.8333 and parameters: {'Hidden_layers': 3, 'neurons_per_layer': 32, 'epochs': 10, 'learning_rate': 0.006323730902627157, 'dropout_rate': 0.4, 'batch_size': 32, 'optimizer': 'RMSprop', 'weight_decay': 0.00012279346511244314}. Best is trial 0 with value: 0.8333.
[I 2025-04-14 05:36:37,542] Trial 1 finished with value: 0.8512 and parameters: {'Hidden_layers': 1, 'neurons_per_layer': 120, 'epochs': 10, 'learning_rate': 0.0009971986279355288, 'dropout_rate': 0.5, 'batch_size': 32, 'optimizer': 'RMSprop', 'weight_decay': 0.0003940811846404622}. Best is trial 1 with value: 0.8512.
[I 2025-04-14 05:37:05,614] Trial 2 finished with value: 0.7146 and parameters: {'Hidden_layers': 1, 'neurons_per_layer': 48, 'epochs': 10, 'learning_rate': 3.110210250112769e-05, 'dropout_rate': 0.30000000000000004, 'batch_size': 16, 'optimizer': 'RMSprop', 'weight_decay': 5.664119464070669e-05}. Best is trial 1 with value: 0.8512.
[I 2025-04-14 05:41:17,408

In [51]:
study.best_value#89.42->88.98(overcomplicated)

0.8898

In [52]:
study.best_params

{'Hidden_layers': 4,
 'neurons_per_layer': 128,
 'epochs': 40,
 'learning_rate': 0.01191854790696021,
 'dropout_rate': 0.4,
 'batch_size': 16,
 'optimizer': 'RMSprop',
 'weight_decay': 0.0001448751455038667}